# 02 - Checks before modelling

Question: would I know every column at the moment of prediction, is the missing data random, and how should train and test be split?

In [2]:
import pandas as pd

df = pd.read_csv('../data.raw/weatherAUS.csv', parse_dates=['Date'])
df = df.sort_values(['Location', 'Date'])

df['next_day_rain'] = df.groupby('Location')['RainToday'].shift(-1)
df['next_date'] = df.groupby('Location')['Date'].shift(-1)

consecutive = (df['next_date'] - df['Date']).dt.days == 1
check = df[consecutive].dropna(subset=['RainTomorrow', 'next_day_rain'])

print(len(check))
print((check['RainTomorrow'] == check['next_day_rain']).mean())


142017
1.0


In [3]:
d = df.drop(columns = ['next_day_rain', 'next_date']).dropna(subset=['RainTomorrow'])

train_end = d['Date'].quantile(0.8)
val_end = d['Date'].quantile(0.9)

train = d[d['Date'] <= train_end]
val = d[(d['Date'] > train_end) & (d['Date'] <= val_end)]
test = d[d['Date'] > val_end]

for name, part in [('train', train), ('val', val), ('test', test)]:
    rain_rate = (part['RainTomorrow'] == 'Yes').mean()
    print(name, len(part), part['Date'].min().date(), part['Date'].max().date(), round(rain_rate, 3))

train 113796 2007-11-01 2015-11-10 0.224
val 14184 2015-11-11 2016-09-01 0.228
test 14213 2016-09-02 2017-06-25 0.218


Finding: RainTomorrow is exactly the next day's RainToday at the same station (142,017 pairs, 100% match). Every feature is measured on the day itself. Missingness follows station equipment, not chance.

Why it matters: No column leaks the answer, but a random split would, because neighbouring days at a station are near-copies.

Decision: Split by date at 80/10/10: train to 2015-11-10, validation to 2016-09-01, test after. Calculate any fill values from train only.


Limit: Validation and test each cover about ten months and different seasons; three stations have under 1,000 training rows.
